**TRANING PROCESSES**

In [46]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import  confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

import pandas as pd
import numpy as np

**READ THE DATA**

In [47]:
df = pd.read_csv("../data/obesity_train.csv")
df.head()

,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE,Gender_Female,Gender_Male,family_history_with_overweight_no,family_history_with_overweight_yes,FAVC_no,FAVC_yes,CAEC_Always,CAEC_Frequently,CAEC_Sometimes,CAEC_no,SMOKE_no,SMOKE_yes,SCC_no,SCC_yes,CALC_Always,CALC_Frequently,CALC_Sometimes,CALC_no,MTRANS_Automobile,MTRANS_Bike,MTRANS_Motorbike,MTRANS_Public_Transportation,MTRANS_Walking,is_obesity
0,37.638102,1.750085,118.114184,2.073224,3.000000,2.024035,0.131371,0.000000,0,1,0,1,0,1,0,0,1,0,1,0,1,0,0,0,1,0,1,0,0,0,0,1
1,20.914366,1.644751,101.067988,2.801992,1.343117,1.128942,0.233987,0.819980,0,1,0,1,0,1,0,0,1,0,1,0,1,0,0,0,0,1,0,0,0,1,0,1
2,20.102241,1.816868,153.959945,3.000000,3.000000,2.414739,1.917014,0.927982,1,0,0,1,0,1,0,0,1,0,1,0,1,0,0,0,1,0,0,0,0,1,0,1
3,20.244359,1.559186,41.952805,2.619835,1.000000,1.198883,0.000000,0.751212,1,0,1,0,0,1,0,1,0,0,1,0,1,0,0,0,1,0,0,0,0,1,0,0
4,22.538932,1.603963,82.012637,1.123672,1.014916,2.042078,1.846666,1.426103,1,0,0,1,0,1,0,0,1,0,1,0,1,0,0,0,0,1,0,0,0,1,0,1


**DEFINE FEATURES AND TARGET**

In [48]:
y = df['is_obesity']
X = df.drop('is_obesity', axis=1)
print("Features shape:")
print(X.shape)
print("Target shape:")
print(y.shape)

Features shape:
(1419, 31)
Target shape:
(1419,)


**SPLIT THE DATA SET**

In [49]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)
print("X_train shape:")
print(X_train.shape)
print("X_val shape:")
print(X_val.shape)
print("y_train shape:")
print(y_train.shape)
print("y_val shape:")
print(y_val.shape)

X_train shape:
(1135, 31)
X_val shape:
(284, 31)
y_train shape:
(1135,)
y_val shape:
(284,)


**SCALE DATA**

In [50]:
standardizer = StandardScaler()
X_train_scaled = standardizer.fit_transform(X_train)
X_val_scaled = standardizer.transform(X_val)
print("X_train shape after standardization:")
print(X_train_scaled.shape)
print("X_val shape after standardization:")
print(X_val_scaled.shape)
print("y_train shape after standardization:")
print(y_train.shape)
print("y_val shape after standardization:")
print(y_val.shape)

X_train shape after standardization:
(1135, 31)
X_val shape after standardization:
(284, 31)
y_train shape after standardization:
(1135,)
y_val shape after standardization:
(284,)


**TRAINING DATA WITH LOGISTIC REGESSION**

In [54]:
# Train a logistic regression model and make predictions on the validation set
model = LogisticRegression()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_val_scaled)

print("Classification report for the validation set:")
print(classification_report(y_val, y_pred))
print("Confusion matrix for the validation set:")
print(confusion_matrix(y_val, y_pred))

Classification report for the validation set:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       152
           1       1.00      1.00      1.00       132

    accuracy                           1.00       284
   macro avg       1.00      1.00      1.00       284
weighted avg       1.00      1.00      1.00       284

Confusion matrix for the validation set:
[[152   0]
 [  0 132]]


In [55]:
# Load the test set
df_test = pd.read_csv("../data/obesity_test.csv")

X_test = df_test.drop(columns=["is_obesity"])
y_test = df_test["is_obesity"]

In [57]:
# Apply the same scaler fitted on the training data
X_test_scaled = standardizer.transform(X_test)

# Predict using scaled test features
y_pred = model.predict(X_test_scaled)

print("Classification report for the test set:")
print(classification_report(y_test, y_pred))

print("Confusion matrix for the test set:")
print(confusion_matrix(y_test, y_pred))

Classification report for the test set:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       190
           1       1.00      0.99      0.99       165

    accuracy                           0.99       355
   macro avg       0.99      0.99      0.99       355
weighted avg       0.99      0.99      0.99       355

Confusion matrix for the test set:
[[190   0]
 [  2 163]]


The logistic regression works well on this data set with the high accuracy score in both validation and test

**USING SVM**

In [58]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

In [61]:
# Train and Validation set are declared above
param_grid = {
    "kernel": ["linear", "rbf"],
    "C": [0.1, 1, 10],
    "gamma": ["scale", "auto", 0.1, 1]
}

grid_search = GridSearchCV(
    estimator=SVC(class_weight="balanced"),
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print("Best parameters found:")
print(grid_search.best_params_)

print("Best cross-validation accuracy:")
print(grid_search.best_score_)

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_val_scaled)

print("Classification report:")
print(classification_report(y_val, y_pred, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_val, y_pred))

Best parameters found:
{'C': 1, 'gamma': 'scale', 'kernel': 'linear'}
Best cross-validation accuracy:
0.9929515418502202
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       152
           1       1.00      1.00      1.00       132

    accuracy                           1.00       284
   macro avg       1.00      1.00      1.00       284
weighted avg       1.00      1.00      1.00       284

Confusion matrix:
[[152   0]
 [  0 132]]


In [62]:
# Perform grid search for the SVM model and evaluate the best model on the test set
y_pred = best_model.predict(X_test_scaled)

print("Classification report for the test set with the best model:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion matrix for the test set with the best model:")
print(confusion_matrix(y_test, y_pred))

Classification report for the test set with the best model:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       190
           1       0.99      0.98      0.98       165

    accuracy                           0.98       355
   macro avg       0.98      0.98      0.98       355
weighted avg       0.98      0.98      0.98       355

Confusion matrix for the test set with the best model:
[[188   2]
 [  4 161]]
